In [5]:
# CMOR demo https://github.com/PCMDI/obs4MIPs-cmor-tables/tree/master/demo
# Test notebook to sese if the demo file can be processed on Mac offline using Jupyter environ.
# Input file pths, .jsons and tables defined for local machine

# import common libraries 
import pandas as pd
import netCDF4 as nc
from netCDF4 import Dataset
import cmor
import xarray as xr
import xcdat as xc
import numpy as np
import json
import sys,os

sys.path.append("/Users/paul.smith/obs4MIPs-cmor-tables/demo/Tables") # Path to obs4MIPsLib used to trap provenance
import obs4MIPsLib

cmorTable = '/Users/paul.smith/obs4MIPs-cmor-tables/demo/Tables/obs4MIPs_Amon.json' ; # Aday,Amon,Lmon,Omon,SImon,fx,monNobs,monStderr - Load target table, axis info (coordinates, grid*) and CVs
inputJson = '/Users/paul.smith/obs4MIPs-cmor-tables/demo/Tables/CMAP-V1902.json' ; # Update contents of this file to set your global_attributes
inputFilePath = '/Users/paul.smith/obs4MIPs-cmor-tables/demo/Data/precip.mon.mean.nc'
inputVarName = 'precip'
outputVarName = 'pr'
outputUnits = 'kg m-2 s-1'

ModuleNotFoundError: No module named 'obs4MIPsLib'

In [7]:
# Open and read input netcdf file, get coordinates and add bounds
f = xc.open_dataset(inputFilePath,decode_times=False)
d = f[inputVarName]
lat = f.lat.values 
lon = f.lon.values 
time = f.time.values  
f = f.bounds.add_missing_bounds(axes=['X', 'Y'])
f = f.bounds.add_bounds("T")
tbds = f.time_bnds.values

In [8]:
# CONVERT UNITS FROM mm/day to kg/m2/s 
d = np.divide(d,86400.)
d = np.where(np.isnan(d),1.e20,d)

In [14]:
# Initialize and run CMOR. For more information see https://cmor.llnl.gov/mydoc_cmor3_api/
cmor.setup(inpath='./',netcdf_file_action=cmor.CMOR_REPLACE_4) #,logfile='cmorLog.txt')
cmor.dataset_json(inputJson)
cmor.load_table(cmorTable)
cmor.set_cur_dataset_attribute('history',f.history) 

0

In [15]:
# Create CMOR axes
cmorLat = cmor.axis("latitude", coord_vals=lat[:], cell_bounds=f.lat_bnds.values, units="degrees_north")
cmorLon = cmor.axis("longitude", coord_vals=lon[:], cell_bounds=f.lon_bnds.values, units="degrees_east")
cmorTime = cmor.axis("time", coord_vals=time[:], cell_bounds=tbds, units= f.time.units)
cmoraxes = [cmorTime,cmorLat, cmorLon]

In [16]:
# Setup units and create variable to write using cmor - see https://cmor.llnl.gov/mydoc_cmor3_api/#cmor_set_variable_attribute
varid   = cmor.variable(outputVarName,outputUnits,cmoraxes,missing_value=1.e20)
values  = np.array(d,np.float32)[:]

In [17]:
# Append valid_min and valid_max to variable before writing using cmor - see https://cmor.llnl.gov/mydoc_cmor3_api/#cmor_set_variable_attribute
cmor.set_variable_attribute(varid,'valid_min','f',2.0)
cmor.set_variable_attribute(varid,'valid_max','f',3.0)

0

In [18]:
# Provenance info - produces global attribute <obs4MIPs_GH_Commit_ID> 
gitinfo = obs4MIPsLib.ProvenanceInfo(obs4MIPsLib.getGitInfo("/Users/paul.smith/demo/Obs4MIPs/misc"))
full_git_path = f"https://github.com/PCMDI/obs4MIPs-cmor-tables/tree/{gitinfo['commit_number']}/demo"  
cmor.set_cur_dataset_attribute("processing_code_location",f"{full_git_path}")

filePath not a valid git-tracked file


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# Prepare variable for writing, then write and close file - see https://cmor.llnl.gov/mydoc_cmor3_api/#cmor_set_variable_attribute
cmor.set_deflate(varid,1,1,1) ; # shuffle=1,deflate=1,deflate_level=1 - Deflate options compress file data
cmor.write(varid,d,len(time)) 
cmor.close()
f.close()